# Metastable Reasoning-Mode Discovery in GPT-Class Models

> **Goal**: Discover and validate metastable reasoning modes (attractors) in transformer activations
> using HMM / spectral clustering on activation trajectories.

> **Key reference**: Rabinovich et al. (2008) *PMC4930057* — metastable attractors in neural systems
> **Method**: Collect activation trajectories → build Markov chain → identify metastable sets → compute AFI

## Pipeline Overview

```
1. [Cell 2] Collect activation trajectories from LLM layers
2. [Cell 3] Project to low-dimensional embedding (PCA / UMAP)
3. [Cell 4] Discretize state space → build transition matrix T
4. [Cell 5] Spectral clustering → discover metastable sets
5. [Cell 6] Validate modes: MFPT, within-cluster coherence
6. [Cell 7] Compute Markov-chain AFI per mode
7. [Cell 8] Visualize metastable partition + transitions
```

**Runtime**: ~30 min for 10K token sequence on a single A100 GPU.

In [ ]:
# Cell 2: Collect activation trajectories
# Run this on your model + diverse reasoning task suite

import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, HookedTransformer
from typing import List, Optional
import json

class ActivationCollector:
    """Collect layer activations from a transformer model."""
    
    def __init__(self, model_name: str, layer_names: Optional[List[str]] = None):
        self.model_name = model_name
        self.activations = {}
        self.layer_names = layer_names or [
            "h.11.mlp.dropout",  # Example for GPT-2
            "h.10.mlp.dropout",
            "h.9.mlp.dropout",
            "h.8.mlp.dropout",
        ]
        
        # Load model
        self.model = HookedTransformer.from_pretrained(model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)
        
        # Register hooks
        self._register_hooks()
    
    def _register_hooks(self):
        """"Register forward hooks on specified layers."""
        def make_hook(name):
            def hook(module, input, output):
                if name not in self.activations:
                    self.activations[name] = []
                # Capture post-activation state
                if hasattr(output, 'to'):
                    self.activations[name].append(output.to("cpu"))
                return output
            return hook
        
        for name in self.layer_names:
            try:
                module = self.model.get_submodule(name)
                module.register_forward_hook(make_hook(name))
            except Exception as e:
                print(f"Warning: layer {name} not found ({e})")
    
    def collect(self, text: str, max_len: int = 512) -> dict:
        """Run model on text, collect all layer activations."""
        # Clear previous activations
        for k in self.activations:
            self.activations[k] = []
        
        # Tokenize
        inputs = self.tokenizer(text, return_tensors="pt", 
                               max_length=max_len, truncation=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            _ = self.model(**inputs)
        
        # Concatenate into trajectory arrays
        result = {}
        for name, acts in self.activations.items():
            if len(acts) > 0:
                # Stack: (n_layers, n_tokens, n_hidden)
                stacked = torch.cat([a.unsqueeze(0) for a in acts], dim=0)
                result[name] = stacked  # shape: (seq_len, n_hidden)
        
        return result


# Example usage:
collector = ActivationCollector("gpt2")  # Or your custom model

tasks = [
    "Prove that the sum of two even numbers is even.",
    "What's the weather like today?",
    "I feel really sad and lonely tonight.",
    "Remember to buy milk on the way home.",
    "Why do cats chase laser pointers?",
    "Calculate 47 * 123 = ?",
    "What's the meaning of life?",
    "Explain quantum entanglement to a 5-year-old.",
    "I'm sorry I yelled at you earlier.",
    "Should I invest in stocks or bonds?",
]

trajectories = []
mode_labels = []  # For ground truth validation (optional)

for task in tasks:
    result = collector.collect(task)
    # Flatten all layer activations into a single trajectory vector
    for layer_name, acts in result.items():
        trajectories.append({
            "task": task[:50],
            "layer": layer_name,
            "activations": acts  # (seq_len, n_hidden)
        })

print(f"Collected {len(trajectories)} trajectories")
print(f"Sample shape: {trajectories[0]['activations'].shape}")

In [ ]:
# Cell 3: Dimensionality reduction — PCA then UMAP

import numpy as np
from sklearn.decomposition import PCA
from umap import UMAP

def stack_trajectories(trajectories):
    """Stack all activation tensors into (n_samples, n_hidden) array."""
    all_vectors = []
    for traj in trajectories:
        acts = traj["activations"]  # (seq_len, n_hidden)
        # Take mean over sequence length to get one vector per task/layer
        if len(acts.shape) == 3:
            acts = acts.mean(dim=0)  # (seq_len, n_hidden) → (n_hidden,)
        all_vectors.append(acts.flatten().numpy())
    return np.array(all_vectors)

def reduce_dimensions(X: np.ndarray, n_components: int = 20) -> np.ndarray:
    """Reduce to n_components via PCA + UMAP."""
    # PCA to remove noise
    pca = PCA(n_components=min(n_components * 2, X.shape[1]))
    X_pca = pca.fit_transform(X)
    print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.3f}")
    
    # UMAP for nonlinear embedding
    umap = UMAP(n_components=n_components, n_neighbors=15, 
                min_dist=0.1, metric="cosine")
    X_embedded = umap.fit_transform(X_pca)
    return X_embedded


X_raw = stack_trajectories(trajectories)
print(f"Raw shape: {X_raw.shape}")

X_embedded = reduce_dimensions(X_raw, n_components=15)
print(f"Embedded shape: {X_embedded.shape}")

In [ ]:
# Cell 4: Discretize state space → build transition matrix

from sklearn.cluster import KMeans
from collections import defaultdict

def discretize(X: np.ndarray, n_bins: int = 50) -> np.ndarray:
    """Discretize continuous embedding into n_bins clusters."""
    kmeans = KMeans(n_clusters=n_bins, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    return labels, kmeans

def build_transition_matrix(
    state_seq: np.ndarray, 
    n_states: int,
    lag: int = 1,
    discount: float = 0.9
) -> np.ndarray:
    """
    Build transition probability matrix from state sequence.
    
    T[i,j] = Pr(X_{t+lag}=j | X_t=i)
    Uses discounting for longer sequences.
    """
    T = np.zeros((n_states, n_states))
    for t in range(len(state_seq) - lag):
        i = state_seq[t]
        j = state_seq[t + lag]
        T[i, j] += 1
    
    # Row-normalize
    row_sums = T.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1  # Avoid division by zero
    T = T / row_sums
    
    return T

def build_trajectory_transitions(
    trajectories: List[dict],
    kmeans,
    lag: int = 1,
) -> tuple[np.ndarray, np.ndarray, list]:
    """
    Build transition matrix from per-task trajectories.
    
    Returns: (T, state_labels, all_states)
    """
    n_bins = kmeans.n_clusters
    all_states = []
    task_state_seqs = []
    
    for traj in trajectories:
        acts = traj["activations"]
        # Project to embedding space
        if len(acts.shape) == 3:
            acts = acts.mean(dim=0)
        flat = acts.flatten().numpy().reshape(1, -1)
        
        # Assign to bins
        states = kmeans.predict(flat)
        all_states.extend(states.tolist())
        task_state_seqs.append(states)
    
    all_states = np.array(all_states)
    T = build_transition_matrix(all_states, n_bins, lag=lag)
    
    return T, all_states, task_state_seqs


# Discretize
state_labels, kmeans = discretize(X_embedded, n_bins=50)
print(f"Discretized into {kmeans.n_clusters} states")

# Build transitions
T, all_states, task_seqs = build_trajectory_transitions(
    trajectories, kmeans, lag=1
)
print(f"Transition matrix shape: {T.shape}")
print(f"Non-zero transitions: {np.count_nonzero(T)}")

In [ ]:
# Cell 5: Spectral clustering → metastable sets

from scipy.linalg import eig
from scipy.sparse.csgraph import laplacian

def spectral_cluster_metastable(T: np.ndarray, n_clusters: int = 6):
    """
    Find metastable sets via spectral clustering of the transition matrix.
    
    Method:
    1. Symmetrize: A = T + T^T
    2. Compute Laplacian L = D - A (D = degree matrix)
    3. Fiedler eigenvector → partition
    4. Refine with k-means on the partition
    
    The partition gives metastable sets.
    """
    # Symmetrize (undirected view of transition graph)
    A = T + T.T
    
    # Degree matrix
    D = np.diag(A.sum(axis=1))
    
    # Normalized Laplacian
    D_inv_sqrt = np.diag(1.0 / np.sqrt(np.diag(D) + 1e-10))
    L_norm = D_inv_sqrt @ (D - A) @ D_inv_sqrt
    
    # Eigenvectors of Laplacian
    eigvals, eigvecs = eig(L_norm)
    
    # Sort by eigenvalue
    idx = np.argsort(eigvals.real)
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]
    
    # Fiedler vector (second smallest eigenvector)
    fiedler = eigvecs[:, 1].real
    
    # Partition based on sign of Fiedler vector
    initial_partition = (fiedler > 0).astype(int)
    
    # Refine with k-means on the eigenvectors
    embedding = eigvecs[:, 1:n_clusters+1].real
    from sklearn.cluster import KMeans
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    metastable_labels = kmeans.fit_predict(embedding)
    
    return metastable_labels, kmeans, eigvals

def compute_metastable_sets(T: np.ndarray, metastable_labels: np.ndarray):
    """Compute statistics for each metastable set."""
    n_clusters = len(set(metastable_labels))
    metastable_stats = {}
    
    for cluster_id in range(n_clusters):
        mask = metastable_labels == cluster_id
        # Sub-matrix for this metastable set
        sub_T = T[np.ix_(mask, mask)]
        
        if sub_T.size > 0 and sub_T.sum() > 0:
            # Row-normalize to get transition probs within set
            row_sums = sub_T.sum(axis=1, keepdims=True)
            sub_T_norm = sub_T / (row_sums + 1e-10)
            
            # Mean self-loop probability (staying in set)
            stay_prob = np.diag(sub_T_norm).mean() if sub_T_norm.shape[0] > 0 else 0
            
            metastable_stats[cluster_id] = {
                "size": mask.sum(),
                "stay_prob": stay_prob,
                "MFPT": 1.0 / (1.0 - stay_prob + 1e-10),  # Mean steps before exit
            }
    
    return metastable_stats


# Discover metastable sets
n_metastable = 6  # Hypothesize 6 reasoning modes
metastable_labels, metastable_kmeans, eigvals = spectral_cluster_metastable(
    T, n_clusters=n_metastable
)
print(f"Eigenvalues (Laplacian): {eigvals.real[:8].round(4)}")

stats = compute_metastable_sets(T, metastable_labels)
for cid, s in stats.items():
    print(f"Metastable set {cid}: size={s['size']}, stay_prob={s['stay_prob']:.3f}, "
          f"MFPT≈{s['MFPT']:.1f} steps")

In [ ]:
# Cell 6: Validate — MFPT and within-cluster coherence

from collections import deque
import numpy as np

def compute_mfpt(T: np.ndarray, from_state: int, to_state: int, 
                  max_steps: int = 1000) -> float:
    """
    Compute mean first passage time from from_state to to_state.
    Uses dynamic programming on the Markov chain.
    """
    n = T.shape[0]
    # Mean first passage time via system of linear equations
    # MFPT(i→j) satisfies: MFPT(i→j) = 1 + Σ_k T[i,k] * MFPT(k→j) for k ≠ j
    # With absorbing state j removed
    
    if from_state == to_state:
        return 0.0
    
    # Build reduced matrix without to_state
    keep = [i for i in range(n) if i != to_state]
    Q = T[np.ix_(keep, keep)]
    
    # Fundamental matrix N = (I - Q)^{-1}
    I = np.eye(len(keep))
    try:
        N = np.linalg.inv(I - Q + 1e-10 * np.eye(len(keep)))
    except np.linalg.LinAlgError:
        return float('inf')
    
    # t = N · 1 (column vector of ones)
    t = N.sum(axis=1)
    
    # Find index of from_state in reduced system
    if from_state > to_state:
        from_idx = from_state - 1
    else:
        from_idx = from_state
    
    return min(t[from_idx], max_steps)


def validate_metastable_partition(T: np.ndarray, 
                                 metastable_labels: np.ndarray,
                                 task_seqs: list,
                                 threshold_mfpt: int = 5):
    """
    Validate metastable partition:
    1. MFPT within each metastable set should be > threshold
    2. Transitions between sets should be rare
    """
    n_clusters = len(set(metastable_labels))
    n_states = T.shape[0]
    
    results = {
        "within_mfpt": {},
        "cross_transitions": {},
        "coherence_score": {},
    }
    
    for c in range(n_clusters):
        mask = metastable_labels == c
        states_in_set = np.where(mask)[0]
        
        if len(states_in_set) < 2:
            continue
        
        # Average within-set MFPT
        mfpts = []
        for i in range(len(states_in_set)):
            for j in range(len(states_in_set)):
                if i != j:
                    mfpt = compute_mfpt(T, states_in_set[i], states_in_set[j])
                    mfpts.append(mfpt)
        
        results["within_mfpt"][c] = np.mean(mfpts) if mfpts else float('inf')
        
        # Cross-set transition rate
        cross = 0.0
        total = 0.0
        for si in states_in_set:
            for sj in range(n_states):
                if not mask[sj]:
                    cross += T[si, sj]
                    total += 1
        
        results["cross_transitions"][c] = cross / total if total > 0 else 0
        results["coherence_score"][c] = results["within_mfpt"][c] / (cross + 1e-10)
    
    return results


validation = validate_metastable_partition(
    T, metastable_labels, task_seqs
)
print("=== Metastable Partition Validation ===")
for cid in sorted(validation["within_mfpt"].keys()):
    print(f"\nMetastable set {cid}:")
    print(f"  Within MFPT: {validation['within_mfpt'][cid]:.2f} steps")
    print(f"  Cross-set transition rate: {validation['cross_transitions'][cid]:.4f}")
    print(f"  Coherence score: {validation['coherence_score'][cid]:.2f}")

In [ ]:
# Cell 7: Compute Markov-Chain AFI per metastable mode

from scipy.linalg import eig
import numpy as np

def compute_stationary_distribution(T: np.ndarray, 
                                    max_iter: int = 1000,
                                    tol: float = 1e-8) -> np.ndarray:
    """Power iteration to find stationary distribution π."""
    n = T.shape[0]
    pi = np.ones(n) / n
    for _ in range(max_iter):
        pi_new = pi @ T
        pi_new = pi_new / pi_new.sum()  # Normalize
        if np.abs(pi_new - pi).max() < tol:
            return pi_new
        pi = pi_new
    return pi

def compute_markov_afi(T: np.ndarray, 
                      metastable_labels: np.ndarray,
                      w_lambda: float = 0.4,
                      w_mu: float = 0.3,
                      w_kappa: float = 0.3) -> dict:
    """
    Compute AFI over Markov chain.
    
    AFI = w_lambda · AFI_lambda + w_mu · AFI_mu + w_kappa · AFI_kappa
    
    where:
    - AFI_lambda: spectral fragility (1 - |lambda_2|, mixing rate)
    - AFI_mu: occupancy imbalance (Gini of stationary distribution)
    - AFI_kappa: condition number of transition graph
    """
    n = T.shape[0]
    
    # 1. Stationary distribution
    pi = compute_stationary_distribution(T)
    
    # 2. Eigenvalues
    eigvals, _ = eig(T)
    eigvals = np.abs(eigvals)  # Magnitudes
    eigvals = np.sort(eigvals)[::-1]
    
    lambda_1 = eigvals[0]  # Should be ≈ 1
    lambda_2 = eigvals[1] if len(eigvals) > 1 else 0
    
    # 3. Mixing time
    if lambda_2 < 1 - 1e-10:
        tau_mix = -1.0 / np.log(lambda_2 + 1e-10)
    else:
        tau_mix = 1e10
    
    AFI_lambda = 1.0 / (tau_mix + 1.0)
    
    # 4. Occupancy imbalance (Gini)
    pi_uniform = 1.0 / n
    gini = np.sum(np.abs(pi - pi_uniform)) / (2 * n * pi_uniform)
    AFI_mu = gini
    
    # 5. Condition number
    try:
        svd = np.linalg.svd(T, compute_uv=False)
        kappa = svd[0] / (svd[-1] + 1e-10)
        kappa = min(kappa, 1e10)  # Cap
        AFI_kappa = np.log10(kappa + 1) / 10.0  # Normalize
    except:
        AFI_kappa = 0.0
    
    # 6. Overall AFI
    AFI = w_lambda * AFI_lambda + w_mu * AFI_mu + w_kappa * AFI_kappa
    
    # 7. Per-mode AFI
    per_mode_afi = {}
    for cluster_id in set(metastable_labels):
        mask = metastable_labels == cluster_id
        pi_cluster = pi[mask].sum()
        # Local second eigenvalue (approximation: use global with rescaling)
        lambda_2_local = lambda_2 * pi_cluster  # Heuristic
        tau_mix_local = -1.0 / np.log(lambda_2_local + 1e-10) if lambda_2_local < 1 - 1e-10 else 1e10
        
        per_mode_afi[cluster_id] = {
            "AFI": w_lambda * (1 - lambda_2_local) + w_mu * (1 - pi_cluster) + w_kappa * AFI_kappa,
            "occupancy": pi_cluster,
            "mixing_time": tau_mix_local,
            "n_states": mask.sum(),
        }
    
    return {
        "AFI": AFI,
        "AFI_lambda": AFI_lambda,
        "AFI_mu": AFI_mu,
        "AFI_kappa": AFI_kappa,
        "lambda_1": lambda_1,
        "lambda_2": lambda_2,
        "tau_mix": tau_mix,
        "stationary_dist": pi,
        "per_mode": per_mode_afi,
    }


afi_results = compute_markov_afi(T, metastable_labels)
print("=== Markov-Chain AFI ===")
print(f"Overall AFI: {afi_results['AFI']:.4f}")
print(f"  AFI_lambda (spectral): {afi_results['AFI_lambda']:.4f}")
print(f"  AFI_mu (occupancy): {afi_results['AFI_mu']:.4f}")
print(f"  AFI_kappa (condition): {afi_results['AFI_kappa']:.4f}")
print(f"  Mixing time τ_mix: {afi_results['tau_mix']:.2f} steps")
print(f"  λ_1: {afi_results['lambda_1']:.4f}, λ_2: {afi_results['lambda_2']:.4f}")
print()
print("Per-mode AFI:")
for cid, m in afi_results["per_mode"].items():
    print(f"  Mode {cid}: AFI={m['AFI']:.4f}, occupancy={m['occupancy']:.3f}, "
          f"states={m['n_states']}, τ_mix={m['mixing_time']:.1f}")

In [ ]:
# Cell 8: Visualize metastable partition + transitions

import matplotlib.pyplot as plt
import numpy as np

def plot_metastable_partition(T, metastable_labels, afi_results, 
                              save_path="metastable_partition.png"):
    """"Visualize the metastable partition and transition graph."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # 1. Stationary distribution by metastable set
    ax = axes[0, 0]
    pi = afi_results["stationary_dist"]
    n_clusters = len(set(metastable_labels))
    colors = plt.cm.Set2(np.linspace(0, 1, n_clusters))
    
    occupancy_by_mode = []
    for cid in range(n_clusters):
        mask = metastable_labels == cid
        occupancy_by_mode.append(pi[mask].sum())
    bars = ax.bar(range(n_clusters), occupancy_by_mode, color=colors)
    ax.set_xlabel("Metastable Mode")
    ax.set_ylabel("Stationary Probability π")
    ax.set_title("Occupancy by Metastable Mode")
    ax.set_xticks(range(n_clusters))
    
    # 2. AFI breakdown per mode
    ax = axes[0, 1]
    per_mode = afi_results["per_mode"]
    afi_vals = [per_mode[c]["AFI"] for c in range(n_clusters)]
    ax.bar(range(n_clusters), afi_vals, color=colors)
    ax.axhline(y=afi_results["AFI"], color="red", linestyle="--", 
               label=f"Overall AFI={afi_results['AFI']:.3f}")
    ax.set_xlabel("Metastable Mode")
    ax.set_ylabel("AFI")
    ax.set_title("Attractor Fragility Index by Mode")
    ax.legend()
    
    # 3. Transition matrix heatmap (clustered)
    ax = axes[1, 0]
    # Sort states by metastable label
    order = np.argsort(metastable_labels)
    T_sorted = T[np.ix_(order, order)]
    im = ax.imshow(T_sorted, cmap="Blues", aspect="auto")
    ax.set_xlabel("To state")
    ax.set_ylabel("From state")
    ax.set_title("Transition Matrix (sorted by metastable set)")
    plt.colorbar(im, ax=ax, label="Transition probability")
    
    # 4. MFPT heatmap for within-set transitions
    ax = axes[1, 1]
    n_clusters = len(set(metastable_labels))
    mfpt_matrix = np.zeros((n_clusters, n_clusters))
    for ci in range(n_clusters):
        for cj in range(n_clusters):
            mask_i = metastable_labels == ci
            mask_j = metastable_labels == cj
            states_i = np.where(mask_i)[0]
            states_j = np.where(mask_j)[0]
            if ci == cj:
                # Within-set: average MFPT
                mfpts = []
                for si in states_i:
                    for sj in states_i:
                        if si != sj:
                            mfpts.append(compute_mfpt(T, si, sj))
                mfpt_matrix[ci, cj] = np.mean(mfpts) if mfpts else 0
            else:
                # Between-set: min MFPT
                mfpts = []
                for si in states_i:
                    for sj in states_j:
                        mfpts.append(compute_mfpt(T, si, sj))
                mfpt_matrix[ci, cj] = np.min(mfpts) if mfpts else float('inf')
    
    im = ax.imshow(mfpt_matrix, cmap="RdYlGn_r", aspect="auto")
    ax.set_xlabel("To metastable mode")
    ax.set_ylabel("From metastable mode")
    ax.set_title("Mean First Passage Time (MFPT)")
    plt.colorbar(im, ax=ax, label="MFPT (steps)")
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"Saved to {save_path}")
    plt.show()


plot_metastable_partition(T, metastable_labels, afi_results)
print("\n=== Interpretation ===")
print("High-AFI modes (red bars in panel 2) are fragile reasoning states")
print("that can be easily disrupted by perturbations.")
print("These should have LOW plasticity in ASRU (more frozen).")
print("Low-AFI modes are robust and can tolerate more exploration.")

In [ ]:
# Cell 9: ASRU integration — export metastable mode parameters

import json

def export_for_asru(afi_results, metastable_labels, stats, 
                     output_path="metastable_modes.json"):
    """
    Export metastable mode parameters for ASRU integration.
    
    format:
    {
      "n_modes": 6,
      "overall_afi": 0.42,
      "modes": {
        "SymbolicManipulation": {
          "id": 0,
          "afi": 0.38,
          "occupancy": 0.12,
          "plasticity_target": 0.3,  // HIGH AFI → low plasticity
          "safety_weight": 0.9,
          "precision_weight": 0.8,
          "column_roles": ["Calculator", "SafetyMonitor", "Sentinel"],
        },
        ...
      }
    }
    """
    mode_names = [
        "SymbolicManipulation",
        "EmotionalResonance",
        "CausalReasoning",
        "AssociativeRecall",
        "Exploratory",
        "SteadyState"
    ]
    
    n_clusters = len(set(metastable_labels))
    
    # Sort by AFI to assign meaningful names
    afi_sorted = sorted(afi_results["per_mode"].items(), 
                        key=lambda x: x[1]["AFI"])
    
    modes_dict = {}
    for rank, (cluster_id, mode_data) in enumerate(afi_sorted):
        name = mode_names[rank] if rank < len(mode_names) else f"Unknown_{rank}"
        
        afi = mode_data["AFI"]
        # High AFI → low plasticity (inverse relationship)
        plasticity_target = max(0.1, min(0.9, 1.0 - afi))
        
        modes_dict[name] = {
            "cluster_id": int(cluster_id),
            "afi": float(afi),
            "occupancy": float(mode_data["occupancy"]),
            "plasticity_target": float(plasticity_target),
            "safety_weight": float(0.9 if afi > 0.5 else 0.5),
            "novelty_weight": float(0.2 if afi > 0.5 else 0.6),
            "mixing_time": float(mode_data["mixing_time"]),
            "n_states": int(mode_data["n_states"]),
        }
    
    output = {
        "n_modes": n_clusters,
        "overall_afi": float(afi_results["AFI"]),
        "tau_mix": float(afi_results["tau_mix"]),
        "modes": modes_dict,
    }
    
    with open(output_path, "w") as f:
        json.dump(output, f, indent=2)
    
    print(f"Exported to {output_path}")
    return output


asru_params = export_for_asru(afi_results, metastable_labels, stats)
print("\n=== ASRU Parameters ===")
for name, params in asru_params["modes"].items():
    print(f"\n{name}:")
    print(f"  AFI: {params['afi']:.4f}")
    print(f"  Plasticity target: {params['plasticity_target']:.2f}")
    print(f"  Occupancy: {params['occupancy']:.3f}")